# 03 — Structured LLM-assisted content analysis with Ollama

This notebook illustrates a reproducible prompt and JSON parsing workflow for a small, manually reviewed sample. **LLM labels are hypotheses, not objective measurements of political orientation.** A media-wide conclusion requires a defensible sample, clear annotation criteria, human validation and uncertainty reporting. Do not send copyrighted text to an external API unless you have the right to do so; this example uses a local Ollama endpoint.

In [ ]:
from pathlib import Path
import json
import sqlite3
import time
import requests
import pandas as pd

DB_PATH = Path("../data/ScrapedNiusData.db")
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.2"
SAMPLE_SIZE = 20  # Start small; review outputs before scaling.
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH.resolve()}")
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist()
    table = "articles" if "articles" in tables else "NiusData"
    columns = pd.read_sql_query(f'PRAGMA table_info("{table}")', conn)["name"].tolist()
    text_column = "article_text" if "article_text" in columns else "text"
    id_column = "id" if "id" in columns else columns[0]
    df_articles = pd.read_sql_query(
        f'SELECT "{id_column}" AS article_id, "{text_column}" AS article_text FROM "{table}" WHERE "{text_column}" IS NOT NULL LIMIT ?',
        conn, params=(SAMPLE_SIZE,)
    )
print(f"Loaded {len(df_articles)} article(s).")


## 1. Prompt design

The prompt asks for a constrained JSON object, separates topic from political framing, requests evidence, and allows an `uncertain` label. This is safer than forcing every article into one of three political categories.

In [ ]:
LABELS = ["right_populist", "left_populist", "neutral_or_non_populist", "uncertain"]
PROMPT_TEMPLATE = """You are assisting a media-content research project. Analyze the text cautiously.
Do not infer a label from topic keywords alone. Distinguish the article's own framing from quoted speech.
If evidence is insufficient or ambiguous, use 'uncertain'. Do not treat this output as a factual verdict.

Return ONLY valid JSON with exactly these keys:
{
  "topic_summary": "short neutral summary",
  "framing_label": "right_populist | left_populist | neutral_or_non_populist | uncertain",
  "genre": "news_report | opinion_commentary | mixed | uncertain",
  "evidence_quotes": ["short exact excerpts from the provided text"],
  "rationale": "brief explanation based only on textual evidence",
  "confidence": "low | medium | high"
}

Article text (may be truncated):
""" 

def build_prompt(article_text: str, max_chars: int = 3000) -> str:
    return PROMPT_TEMPLATE + str(article_text)[:max_chars]


## 2. Call local Ollama and validate the response

Make sure Ollama is running and the model is installed (`ollama pull llama3.2`). The function retries transient failures and validates the JSON structure.

In [ ]:
REQUIRED_KEYS = {"topic_summary", "framing_label", "genre", "evidence_quotes", "rationale", "confidence"}

def analyze_with_ollama(article_text: str, model: str = MODEL, retries: int = 2) -> dict:
    last_error = None
    for attempt in range(retries + 1):
        try:
            response = requests.post(
                OLLAMA_URL,
                json={"model": model, "prompt": build_prompt(article_text), "stream": False, "format": "json"},
                timeout=120,
            )
            response.raise_for_status()
            raw = response.json().get("response", "")
            result = json.loads(raw)
            if not REQUIRED_KEYS.issubset(result):
                raise ValueError("LLM response is missing required JSON keys")
            if result["framing_label"] not in LABELS:
                result["framing_label"] = "uncertain"
            return result
        except (requests.RequestException, json.JSONDecodeError, ValueError, TypeError) as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(2 * (attempt + 1))
    return {"topic_summary":"", "framing_label":"uncertain", "genre":"uncertain",
            "evidence_quotes":[], "rationale":f"Analysis failed: {last_error}", "confidence":"low"}

# Test connection before analysis.
# response = requests.get("http://localhost:11434/api/tags", timeout=5)
# response.raise_for_status()


## 3. Run a small sample

Run this cell only after confirming the local model is available. LLM output can vary between runs; record model/version and settings for formal research.

In [ ]:
# Uncomment to run:
# results = []
# for row in df_articles.itertuples(index=False):
#     result = analyze_with_ollama(row.article_text)
#     results.append({"article_id": row.article_id, **result})
#     time.sleep(0.5)
# df_results = pd.DataFrame(results)
# display(df_results.head())


## 4. Export results for human review

Keep the generated results local by default. Review a sample manually and document disagreements before summarizing labels.

In [ ]:
# Uncomment after running the analysis:
# output_dir = Path("../outputs/tables")
# output_dir.mkdir(parents=True, exist_ok=True)
# df_results.to_csv(output_dir / "llm_content_analysis.csv", index=False)
# df_results["framing_label"].value_counts(dropna=False).to_csv(output_dir / "llm_label_counts.csv")


## Limitations and validation checklist

- The model may misread irony, quotation, context or political terminology.
- Topic words such as migration or taxes are not sufficient evidence of a political frame.
- A single model and prompt do not establish reliability; compare with human-coded examples.
- Report the sample size, collection window, missing records and uncertainty.
- Do not generalize from a convenience sample to the entire publication.
- Store only outputs you are allowed to publish; review exact quotations for copyright concerns.